# PCA via SVD

## Learning Objectives

1. **Derive** the connection between PCA and SVD
2. **Implement** PCA via mean-centering and SVD
3. **Explain** principal components as directions of maximum variance
4. **Compute** variance explained and dimensionality selection

## PCA and SVD: The Connection

**PCA** finds directions (principal components) of maximum variance in mean-centered data.

Let $X \in \mathbb{R}^{n \times d}$ (n data points, d features). Center: $\tilde{X} = X - \bar{x}\mathbf{1}^\top$.

The covariance matrix is $C = \frac{1}{n} \tilde{X}^\top \tilde{X}$.

**Eigendecomposition of $C$:** $C = V \Lambda V^\top$ where columns of $V$ are principal components.

**SVD of $\tilde{X}$:** $\tilde{X} = U \Sigma V^\top$

Comparing: $\tilde{X}^\top \tilde{X} = V \Sigma^2 U^\top U \Sigma V^\top = V \Sigma^2 V^\top$

So **the right singular vectors $V$ of $\tilde{X}$ are the principal components of $X$**,
and **eigenvalues $\lambda_i = \sigma_i^2 / n$**.

In [ ]:
import math, random

def mean_center(X):
    """Subtract column means."""
    n, d = len(X), len(X[0])
    means = [sum(X[i][j] for i in range(n))/n for j in range(d)]
    return [[X[i][j]-means[j] for j in range(d)] for i in range(n)], means

def transpose(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def mat_vec(A, v):
    return [sum(A[i][j]*v[j] for j in range(len(v))) for i in range(len(A))]

def norm(v): return math.sqrt(sum(x**2 for x in v))
def normalize(v): n=norm(v); return [x/n for x in v] if n>0 else v
def dot(a,b): return sum(x*y for x,y in zip(a,b))

def svd_via_power(X, k=2, n_iter=200):
    """Top-k right singular vectors of X via power iteration on X^T X."""
    Xt = transpose(X)
    XtX = [[sum(Xt[i][l]*X[l][j] for l in range(len(X))) for j in range(len(Xt[0]))]
           for i in range(len(Xt))]
    rng = random.Random(1)
    d = len(XtX)
    components = []
    residual = [row[:] for row in XtX]
    for _ in range(k):
        v = normalize([rng.gauss(0,1) for _ in range(d)])
        for _ in range(n_iter):
            v = normalize(mat_vec(residual, v))
        eigenval = dot(v, mat_vec(residual, v))
        components.append((math.sqrt(max(eigenval,0)), v))
        for i in range(d):
            for j in range(d):
                residual[i][j] -= eigenval * v[i] * v[j]
    return components

# 2D dataset with strong first PC direction
rng = random.Random(42)
n, d = 200, 3
X = []
for _ in range(n):
    t = rng.gauss(0, 3)
    noise = [rng.gauss(0, 0.3) for _ in range(d)]
    # Data lives near the [1,2,1] direction
    x = [t + noise[0], 2*t + noise[1], t + noise[2]]
    X.append(x)

Xc, means = mean_center(X)
print(f"Data: {n} points in {d}D")
print(f"Column means: {[round(m,3) for m in means]}")

comps = svd_via_power(Xc, k=2)
total_var = sum(c[0]**2 for c in comps) + 1e-8  # approximate (only top-k)

print("
Principal components:")
for i, (sigma, v) in enumerate(comps):
    print(f"  PC{i+1}: direction={[round(x,3) for x in v]}, σ={sigma:.3f}, σ²={sigma**2:.3f}")

In [ ]:
# Variance explained
n = len(Xc)
total_variance = sum(sum(Xc[i][j]**2 for j in range(len(Xc[0]))) for i in range(n)) / n
print(f"Total data variance: {total_variance:.3f}")
for i,(sigma,v) in enumerate(comps):
    var_explained = sigma**2 / n / total_variance
    print(f"  PC{i+1} explains {var_explained*100:.1f}% of variance")

# Project data onto PC1 and PC2
def project(Xc, components):
    scores = []
    for point in Xc:
        coords = [dot(point, v) for _, v in components]
        scores.append(coords)
    return scores

scores = project(Xc, comps)

# Show first 5 projected points
print("
First 5 data points in PC space:")
print(f"  {'PC1':>8} {'PC2':>8}")
for s in scores[:5]:
    print(f"  {s[0]:>8.3f} {s[1]:>8.3f}")


## PCA vs SVD Summary

| | PCA | SVD |
|--|--|--|
| Input | Mean-centered data $\tilde{X}$ | Any matrix $A$ |
| Components | Eigenvectors of $\tilde{X}^\top \tilde{X}$ | Right singular vectors $V$ |
| Variance | $\lambda_i = \sigma_i^2 / n$ | $\sigma_i$ = singular values |
| Implementation | Eigendecompose covariance | `numpy.linalg.svd(X_centered)` |

**Best practice:** use SVD directly on the centered data matrix (more numerically stable than forming $X^\top X$ explicitly).